In [1]:
import numpy as np
from tabpfn import TabPFNClassifier
from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier

In [2]:
file = "/home/server/Projects/data/AKI/tabular_combined.npz"

# Load data
with np.load(file, allow_pickle=True) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_binary_train = data["y_binary_train"]
    y_binary_test = data["y_binary_test"]

In [ ]:
from tabpfn_extensions.rf_pfn import RandomForestTabPFNClassifier
from tabpfn_extensions import TabPFNClassifier

clf_base = TabPFNClassifier(
    ignore_pretraining_limits=True,
    inference_config = {"SUBSAMPLE_SAMPLES": 2000} # Needs to be set low so that not OOM on fitting intermediate nodes
)

tabpfn_tree_clf = RandomForestTabPFNClassifier(
    tabpfn=clf_base,
    verbose=1,
    max_predict_time=60, # Will fit for one minute
    fit_nodes=True, # Wheather or not to fit intermediate nodes
    adaptive_tree=True, # Whather or not to validate if adding a leaf helps or not
    n_jobs=1
  )

In [4]:
# import torch
# import gc

# del tabpfn_tree_clf  # Replace 'model' with your variable names
# gc.collect()
# torch.cuda.empty_cache()

In [5]:

from sklearn.model_selection import cross_val_score, KFold

cv = KFold(random_state=42, n_splits=3, shuffle=True)
scores_raw_class = cross_val_score(tabpfn_tree_clf, X_train, y_binary_train, cv=cv, scoring='roc_auc', verbose=1)
scores_class = scores_raw_class.mean()

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   21.8s


Starting tree leaf validation train (47018, 515) valid (11755, 515)
Estimators: 1, Nodes per estimator: 37
Leaf: 0 Shape: (47018, 515) Shape test: 11755 / (11755, 515) N classes (Train Test) 2 2 /  2


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 47018 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:1000: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/

Starting tree leaf validation train (47018, 515) valid (11755, 515)
Estimators: 1, Nodes per estimator: 45
Leaf: 0 Shape: (47018, 515) Shape test: 11755 / (11755, 515) N classes (Train Test) 2 2 /  2


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 47018 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:1000: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/

Starting tree leaf validation train (47019, 515) valid (11755, 515)
Estimators: 1, Nodes per estimator: 35
Leaf: 0 Shape: (47019, 515) Shape test: 11755 / (11755, 515) N classes (Train Test) 2 2 /  2


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 47019 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:1000: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/

In [6]:
scores_raw_class

array([nan, nan, nan])

In [7]:
tabpfn_subsample_clf = TabPFNClassifier(
    ignore_pretraining_limits=True,  # (bool) Allows the use of datasets larger than pretraining limits.
    n_estimators=32,                 # (int) Number of estimators for ensembling; improves accuracy with higher values.
    inference_config={
        "SUBSAMPLE_SAMPLES": 2000  # (int) Maximum number of samples per inference step to manage memory usage.
    },
)

In [8]:
cv = KFold(random_state=42, n_splits=3, shuffle=True)
scores_raw_class = cross_val_score(tabpfn_subsample_clf, X_train, y_binary_train, cv=cv, scoring='roc_auc', verbose=1)
scores_class = scores_raw_class.mean()

/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of features 515 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 58773 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:1000: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/